In [ ]:
# 1 - data cleaning
# This notebooks purpose is to fix this issue: several panels appearing in data and data_new_2 
# have different leveling scores.
# We will take this for the ground truth labels: https://swcompany.sharepoint.com/:x:/r/sites/TestMethodsTeam/_layouts/15/Doc.aspx?sourcedoc=%7B00F28E6C-EBF5-4D0B-BCB6-D8443E3BDBCB%7D&file=Leveling%20data%20-%20Avg%20of%203%20sets%204.29.25.xlsx&fromShare=true&action=default&mobileredirect=true
# Changes in place: load_metadata is now updated to reflect the changes in the sheet above

In [23]:
# imports

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

import sys
import os
import importlib
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.build_metadata import build_df_from_data, build_df_from_new_data
from src.data import load_images_from_metadata, filter_images
from src.model import train_random_forest_model

In [24]:
# load all of the data into metadata csv file including the new data (data_new_2)), this will also
# fix the labelling issue (load_metadata is updated)
from src.build_metadata import load_metadata
import importlib
import src.build_metadata as build_metadata

importlib.reload(build_metadata)
build_metadata.load_metadata(
    data_base_dir="../data",
    new_data_base_dir="../data_new",
    new_data_2_base_dir="../data_new_2",
    output_file="metadata.csv",
)

Applied human ratings to 738 DD rows.
Saved metadata to ../data\metadata.csv
Combined records: 1340


,LevelingScore,Person,DateCollected,ImageType,GSCamera,PanelID,State,FilePath
0,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (1...
1,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (2...
2,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (3...
3,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (4...
4,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (5...
...,...,...,...,...,...,...,...,...
1335,10.0,Nicole,7.9.2026,DD,2BDR-9F02,25-154,raw,../data_new_2\10\{leveling-10}_P{Nicole}_D{7.9...
1336,10.0,Nicole,7.9.2026,DD,2BDR-9F02,25-154,raw,../data_new_2\10\{leveling-10}_P{Nicole}_D{7.9...
1337,10.0,Nicole,7.9.2026,DD,2BDR-9F02,25-154,flat,../data_new_2\10\{leveling-10}_P{Nicole}_D{7.9...
1338,10.0,Nicole,7.9.2026,DD,2BDR-9F02,25-154,flat,../data_new_2\10\{leveling-10}_P{Nicole}_D{7.9...


In [25]:
# get the data
data = load_images_from_metadata("../data/metadata.csv", crop_fraction = .4, use_cv2=True)

data_standards = filter_images(data, ImageType = "STD", State = "flat", DateCollected = ["6.2.2026", "6.3.2026", "6.5.2026", "6.8.2026"])
data_real_paint = filter_images(data, ImageType = "DD", State = "flat")

Loaded 1340 images (cropping=ON, backend=cv2)
Filtered down to 177 images
Filtered down to 432 images


In [26]:
# inspect the values for PanelID in the metadata.csv table to confirm which panels we actually have data for
# and that their formats match

In [27]:
import pandas as pd

# Load the metadata
df = pd.read_csv("../data/metadata.csv")

# Count occurrences of each PanelID
panel_counts = df["PanelID"].value_counts().sort_index()

print(f"Total unique panels: {len(panel_counts)}")
print("\nPanel ID counts:")
print(panel_counts)

Total unique panels: 52

Panel ID counts:
PanelID
25-100    32
25-101    18
25-102    12
25-103    10
25-104    12
25-105    18
25-106    12
25-107    18
25-108    18
25-109    12
25-110    18
25-111    18
25-112    12
25-113    12
25-114    12
25-115    18
25-116    12
25-117    12
25-118    12
25-119    12
25-120    18
25-121    12
25-122    12
25-123    12
25-124    18
25-125    12
25-126    12
25-127    12
25-128    18
25-129    12
25-130    12
25-131    12
25-132    16
25-133    12
25-134    18
25-135    18
25-136    18
25-137    12
25-138    12
25-139    12
25-140    12
25-141    12
25-142    18
25-143    12
25-144    18
25-145    18
25-146    14
25-147    18
25-148    12
25-149    12
25-150    12
25-154     6
Name: count, dtype: int64


In [28]:
#
# insight: all of our panels IDs are present in the spreadsheet expect 1, panel 154
# Messaged in the group to check that a label of 10 is correct for this panel.
#

In [29]:
# inspect the datatype of the panelID column
print(df["PanelID"].dtype)

str


In [30]:
# inspect the unique values for PanelID for data_real_paint and data_standards
import pandas as pd

# Extract unique PanelIDs from data_real_paint, filtering out NaN
real_paint_panels = set(d["PanelID"] for d in data_real_paint if pd.notna(d["PanelID"]))
print(f"Unique PanelIDs in data_real_paint: {len(real_paint_panels)}")
print(sorted(real_paint_panels, key=str))

print("\n" + "="*70 + "\n")

# Extract unique PanelIDs from data_standards, filtering out NaN
standards_panels = set(d["PanelID"] for d in data_standards if pd.notna(d["PanelID"]))
print(f"Unique PanelIDs in data_standards: {len(standards_panels)}")
print(sorted(standards_panels, key=str))

print("\n" + "="*70 + "\n")

Unique PanelIDs in data_real_paint: 52
['25-100', '25-101', '25-102', '25-103', '25-104', '25-105', '25-106', '25-107', '25-108', '25-109', '25-110', '25-111', '25-112', '25-113', '25-114', '25-115', '25-116', '25-117', '25-118', '25-119', '25-120', '25-121', '25-122', '25-123', '25-124', '25-125', '25-126', '25-127', '25-128', '25-129', '25-130', '25-131', '25-132', '25-133', '25-134', '25-135', '25-136', '25-137', '25-138', '25-139', '25-140', '25-141', '25-142', '25-143', '25-144', '25-145', '25-146', '25-147', '25-148', '25-149', '25-150', '25-154']


Unique PanelIDs in data_standards: 0
[]




In [31]:
#
# insight: data looks clean for PanelID in the metadata.csv table
#

In [ ]:
import pandas as pd
import re

# Load current metadata and human ratings
metadata_check = pd.read_csv("../data/metadata.csv")
human_ratings = pd.read_csv("../human_ratings/HumanRatings.csv")
part_to_avg = dict(zip(human_ratings["Label"].astype(int), human_ratings["AverageScore"]))

def extract_part_number(panel_id):
    if pd.isna(panel_id):
        return None
    match = re.search(r"25-(\d+)", str(panel_id))
    return int(match.group(1)) if match else None

# Only test DD rows — STD intentionally keeps integer scores
dd_rows = metadata_check[metadata_check["ImageType"] == "DD"].copy()

failures = []
for _, row in dd_rows.iterrows():
    part_num = extract_part_number(row["PanelID"])
    if part_num is not None and part_num in part_to_avg:
        expected = part_to_avg[part_num]
        if round(row["LevelingScore"], 6) != round(expected, 6):
            failures.append({
                "PanelID": row["PanelID"],
                "FilePath": row["FilePath"],
                "Actual": row["LevelingScore"],
                "Expected": expected,
            })

print("=" * 70)
print("TEST: DD panel LevelingScores match HumanRatings.csv")
print("=" * 70)

if failures:
    print(f"❌ FAIL: {len(failures)} rows have incorrect scores:\n")
    print(pd.DataFrame(failures).to_string(index=False))
else:
    total_checked = dd_rows["PanelID"].apply(extract_part_number).map(part_to_avg).notna().sum()
    print(f"✅ PASS: All {total_checked} DD rows match their expected human rating scores.")


TEST: DD panel LevelingScores match HumanRatings.csv
✅ PASS: All 738 DD rows match their expected human rating scores.


: 